In [1]:
import pandas as pd
import os

cleaned_folder = r"C:\Users\Siddarth.b\OneDrive\Documents\DataScience_guvi\Cart2Insights_project1\data\cleaned"

orders = pd.read_csv(os.path.join(cleaned_folder, "orders_cleaned.csv"))
order_items = pd.read_csv(os.path.join(cleaned_folder, "order_items_cleaned.csv"))
order_payments = pd.read_csv(os.path.join(cleaned_folder, "order_payments_cleaned.csv"))
customers = pd.read_csv(os.path.join(cleaned_folder, "customers_cleaned.csv"))
sellers = pd.read_csv(os.path.join(cleaned_folder, "sellers_cleaned.csv"))
products = pd.read_csv(os.path.join(cleaned_folder, "products_cleaned.csv"))

print("Data loaded successfully!")

Data loaded successfully!


In [2]:
order_items["item_total"] = (
    order_items["price"] + order_items["freight_value"]
)

order_value = (
    order_items.groupby("order_id")["item_total"]
    .sum()
    .reset_index()
)

order_value.rename(
    columns={"item_total": "total_order_value"},
    inplace=True
)

print(order_value.head())

                           order_id  total_order_value
0  00010242fe8c5a6d1ba2dd792cb16214              72.19
1  00018f77f2f0320c557190d7a144bdd3             259.83
2  000229ec398224ef6ca0657da4fc703e             216.87
3  00024acbcdf0a6daa1e931b038114c75              25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9             218.04


In [3]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

print(
    orders[
        ["order_id", "order_purchase_timestamp",
         "order_delivered_customer_date", "delivery_days"]
    ].head()
)

                           order_id order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33   
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:37   
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:49   
3  949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:06   
4  ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:39   

  order_delivered_customer_date  delivery_days  
0           2017-10-10 21:25:13            8.0  
1           2018-08-07 15:27:45           13.0  
2           2018-08-17 18:06:29            9.0  
3           2017-12-02 00:28:42           13.0  
4           2018-02-16 18:17:02            2.0  


In [4]:
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

print(
    orders[
        ["order_id",
         "order_delivered_customer_date",
         "order_estimated_delivery_date",
         "delivery_delay_days"]
    ].head()
)

                           order_id order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:13   
1  53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:45   
2  47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:29   
3  949d5b44dbf5de918fe9c16f97b45f8a           2017-12-02 00:28:42   
4  ad21c59c0840e6cb83a9ceb5573f8159           2018-02-16 18:17:02   

  order_estimated_delivery_date  delivery_delay_days  
0                    2017-10-18                 -8.0  
1                    2018-08-13                 -6.0  
2                    2018-09-04                -18.0  
3                    2017-12-15                -13.0  
4                    2018-02-26                -10.0  


In [5]:
customer_order_count = (
    orders.groupby("customer_id")["order_id"]
    .count()
    .reset_index()
)

customer_order_count.rename(
    columns={"order_id": "customer_order_count"},
    inplace=True
)

print(customer_order_count.head())

                        customer_id  customer_order_count
0  00012a2ce6f8dcda20d059ce98491703                     1
1  000161a058600d5901f007fab4c27140                     1
2  0001fd6190edaaf884bcaf3d49edf079                     1
3  0002414f95344307404f0ace7a26f1d5                     1
4  000379cdec625522490c315e70c7a9fb                     1


In [6]:
customer_spending = (
    order_value
    .merge(
        orders[["order_id", "customer_id"]],
        on="order_id",
        how="left"
    )
    .groupby("customer_id")["total_order_value"]
    .sum()
    .reset_index()
)

customer_spending.rename(
    columns={"total_order_value": "customer_total_spending"},
    inplace=True
)

print(customer_spending.head())

                        customer_id  customer_total_spending
0  00012a2ce6f8dcda20d059ce98491703                   114.74
1  000161a058600d5901f007fab4c27140                    67.41
2  0001fd6190edaaf884bcaf3d49edf079                   195.42
3  0002414f95344307404f0ace7a26f1d5                   179.35
4  000379cdec625522490c315e70c7a9fb                   107.01


In [7]:
customer_aov = (
    customer_spending
    .merge(
        customer_order_count,
        on="customer_id",
        how="left"
    )
)

customer_aov["customer_aov"] = (
    customer_aov["customer_total_spending"]
    / customer_aov["customer_order_count"]
)

print(
    customer_aov[
        ["customer_id", "customer_total_spending",
         "customer_order_count", "customer_aov"]
    ].head()
)

                        customer_id  customer_total_spending  \
0  00012a2ce6f8dcda20d059ce98491703                   114.74   
1  000161a058600d5901f007fab4c27140                    67.41   
2  0001fd6190edaaf884bcaf3d49edf079                   195.42   
3  0002414f95344307404f0ace7a26f1d5                   179.35   
4  000379cdec625522490c315e70c7a9fb                   107.01   

   customer_order_count  customer_aov  
0                     1        114.74  
1                     1         67.41  
2                     1        195.42  
3                     1        179.35  
4                     1        107.01  


In [8]:
seller_revenue = (
    order_items
    .groupby("seller_id")["price"]
    .sum()
    .reset_index()
)

seller_revenue.rename(
    columns={"price": "seller_revenue"},
    inplace=True
)

print(seller_revenue.head())

                          seller_id  seller_revenue
0  0015a82c2db000af6aaaf3ae2ecb0532         2685.00
1  001cca7ae9ae17fb1caed9dfb1094831        25080.03
2  001e6ad469a905060d959994f1b41e4f          250.00
3  002100f778ceb8431b7a1020ff7ab48f         1234.50
4  003554e2dce176b5555353e4f3555ac8          120.00


In [9]:
seller_order_count = (
    order_items
    .groupby("seller_id")["order_id"]
    .nunique()
    .reset_index()
)

seller_order_count.rename(
    columns={"order_id": "seller_order_count"},
    inplace=True
)

print(seller_order_count.head())

                          seller_id  seller_order_count
0  0015a82c2db000af6aaaf3ae2ecb0532                   3
1  001cca7ae9ae17fb1caed9dfb1094831                 200
2  001e6ad469a905060d959994f1b41e4f                   1
3  002100f778ceb8431b7a1020ff7ab48f                  51
4  003554e2dce176b5555353e4f3555ac8                   1


In [10]:
customer_order_count["repeat_customer"] = (
    customer_order_count["customer_order_count"] > 1
)

print(
    customer_order_count[
        ["customer_id", "customer_order_count", "repeat_customer"]
    ].head(10)
)

                        customer_id  customer_order_count  repeat_customer
0  00012a2ce6f8dcda20d059ce98491703                     1            False
1  000161a058600d5901f007fab4c27140                     1            False
2  0001fd6190edaaf884bcaf3d49edf079                     1            False
3  0002414f95344307404f0ace7a26f1d5                     1            False
4  000379cdec625522490c315e70c7a9fb                     1            False
5  0004164d20a9e969af783496f3408652                     1            False
6  000419c5494106c306a97b5635748086                     1            False
7  00046a560d407e99b969756e0b10f282                     1            False
8  00050bf6e01e69d5c0fd612f1bcfb69c                     1            False
9  000598caf2ef4117407665ac33275130                     1            False
